# 03 -- Random-Init vs Trained Trace Comparison

Loads the random-initialization control (Phase 2): PyHessian trace of `conv1` at random
init (3 seeds) vs the trained FP32 model, for all three architectures. Reproduces Table 3
(Sec. 5.3) and an optional depth-profile plot showing the random-init trace elevation vs
the trained trace elevation across layer depth for ResNet-50 (the architecture with the
most layers, hence the clearest depth trend).

Source CSVs: `results/20260816_230437_38678/csv/{random_init_comparison,random_init_summary}.csv`


In [1]:
# Requirements: pandas==3.0.5, numpy==2.5.1, matplotlib==3.11.1, seaborn==0.13.2, scipy==1.18.0
# All notebooks in this report use the same environment; paths below are relative to
# report/notebooks/, so the notebook must be run with its own directory as the working
# directory (the default for `jupyter nbconvert --execute` and for Jupyter's own kernel).
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPO = "../.."  # report/notebooks -> report -> repo root
FIG_DIR = "../figures"
import os
os.makedirs(FIG_DIR, exist_ok=True)


In [2]:
RUN = f"{REPO}/results/20260816_230437_38678/csv"
comparison = pd.read_csv(f"{RUN}/random_init_comparison.csv")
summary = pd.read_csv(f"{RUN}/random_init_summary.csv")
comparison.head()


,model,dataset,layer,n_seeds,trace_random_mean,trace_random_std,trace_trained_fp32,ratio_trained_over_random,elev_over_median_random,elev_over_median_trained,classification
0,resnet50_no_weights,CIFAR10,conv1.weight,3,74.428225,68.577994,15.551487,0.208946,0.102096,1.023408,learned
1,resnet50_no_weights,CIFAR10,fc.weight,3,382734.261990,370923.306225,6.439417,0.000017,525.011697,0.423764,architectural
2,resnet50_no_weights,CIFAR10,layer1.0.conv1.weight,3,15.420853,13.647422,1.267965,0.082224,0.021153,0.083442,learned
3,resnet50_no_weights,CIFAR10,layer1.0.conv2.weight,3,142.494339,127.474390,1.310764,0.009199,0.195465,0.086258,architectural
4,resnet50_no_weights,CIFAR10,layer1.0.conv3.weight,3,69.725038,59.935836,0.949744,0.013621,0.095645,0.062501,architectural


Filter to conv1 rows only (report Table 3 is specifically about conv1, the layer central to the Phase-4 predictor analysis).


In [3]:
# conv1.weight is the layer the main predictor analysis centers on; filter to it here
# rather than reporting the full per-layer table (which is used later for the depth plot).
conv1 = comparison[comparison["layer"] == "conv1.weight"].copy()
MODEL_LABEL = {"cnn": "CNN", "resnet18_no_weights": "ResNet-18", "resnet50_no_weights": "ResNet-50"}
conv1["Modell"] = conv1["model"].map(MODEL_LABEL)
conv1["Random"] = conv1.apply(lambda r: f"{r.trace_random_mean:.4g} ± {r.trace_random_std:.4g}", axis=1)
conv1["Klasse"] = conv1["classification"].map({"learned": "gelernt", "architectural": "architektonisch", "mixed": "gemischt"})

table3 = conv1[["Modell", "Random", "trace_trained_fp32", "Klasse"]].copy()
table3.columns = ["Modell", "Random (µ±σ)", "Trainiert", "Klasse"]
table3["Trainiert"] = table3["Trainiert"].round(2)
table3.to_csv(f"{FIG_DIR}/tab_03_random_init_comparison.csv", index=False)
table3


,Modell,Random (µ±σ),Trainiert,Klasse
0,ResNet-50,74.43 ± 68.58,15.55,gelernt
54,ResNet-18,7.217 ± 1.19,17.35,gelernt
75,CNN,0.002393 ± 9.91e-05,65.87,gelernt


Depth-profile plot for ResNet-50: elevation-over-median-trace at random init vs after training, across all layers ordered by their position in the network (proxy for depth via row order in the CSV, which follows model definition order).


In [4]:
# ResNet-50 has the most layers (54) and is the only model with a significant
# random-vs-trained profile correlation (rho=0.60, p<1e-5, per random_init_summary.csv),
# making it the clearest case for a depth-trend plot.
r50 = comparison[comparison["model"] == "resnet50_no_weights"].reset_index(drop=True)
r50["depth_index"] = np.arange(len(r50))  # CSV row order follows model definition order

fig, ax = plt.subplots(figsize=(6.0, 3.0), layout="constrained")
ax.plot(r50["depth_index"], r50["elev_over_median_random"], marker="o", ms=3,
        label="Random-Init", color="#4C72B0")
ax.plot(r50["depth_index"], r50["elev_over_median_trained"], marker="s", ms=3,
        label="Trainiert (FP32)", color="#DD8452")
ax.set_yscale("log")
ax.set_xlabel("Schicht-Index (Tiefe, Definitionsreihenfolge)")
ax.set_ylabel("Elevation über Median-Trace (log)")
ax.set_title("ResNet-50: Trace-Elevation nach Tiefe, Random-Init vs. trainiert")
ax.legend(fontsize=8, frameon=False)
fig.savefig(f"{FIG_DIR}/fig_opt_random_init_depth_profile.pdf")
fig.savefig(f"{FIG_DIR}/fig_opt_random_init_depth_profile.png", dpi=200)
plt.close(fig)


## Output

- `figures/tab_03_random_init_comparison.csv` -- report Table 3 (conv1 only)
- `figures/fig_opt_random_init_depth_profile.pdf` / `.png` -- optional ResNet-50 depth-profile plot (Sec. 5.3)
